# Data Preprocessing

The **objective** of this notebook is to transform the raw retail
transaction data into clean, consistent, reliable, and analysis-ready datasets.

The preprocessing decisions are based on the data validation
performed in Notebook 01.

## Main preprocessing tasks

1. Remove exact duplicate records
2. Identify and exclude cancelled transactions from sales analysis
3. Handle invalid quantities and prices
4. Create transaction-level Sales (Sales = Quantity × UnitPrice)
5. Create separate customer-analysis data
6. Validate the cleaned datasets
7. Export cleaned datasets

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Project paths
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_FILE = RAW_DIR / "Online Retail.xlsx"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# load raw data
df_raw = pd.read_excel(RAW_FILE)

print(f"Raw dataset shape: {df_raw.shape}")

Raw dataset shape: (541909, 8)


In [4]:
# Create working copy
df = df_raw.copy()

In [6]:
# Remove exact duplicates
duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows identified: {duplicate_count:,}")

df = df.drop_duplicates().copy()
print(f"Shape after duplicate removal: {df.shape}")

Exact duplicate rows identified: 5,268
Shape after duplicate removal: (536641, 8)


```
Original df
    ↓
drop duplicate rows
    ↓
create independent copy
    ↓
assign it back to df
```

In [7]:
# Standardize text columns
# eg : " United Kingdom " --> "United Kingdom"

text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [12]:
# Create Sales
df['Sales'] = df['Quantity'] * df['UnitPrice']
df[['Quantity','UnitPrice','Sales']].head()

,Quantity,UnitPrice,Sales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


#### Create two conceptual datasets. 

Missing ```CustomerID``` doesn't necessarily mean the transaction is useless for sales analysis.
But it does prevent customer-level analysis.

**Dataset 1 — Sales analysis**

Used for:
- Revenue
- Product analysis
- Time analysis
- Geographic analysis

**Dataset 2 — Customer analysis**

Used for:
- Customer behavior
- RFM
- Segmentation

In [15]:
# Identify cancelled transactions
df["IsCancelled"] = df["InvoiceNo"].astype("string").str.startswith("C")
df["IsCancelled"].value_counts()

IsCancelled
False    527390
True       9251
Name: count, dtype: Int64

In [16]:
# Create sales-analysis dataset
sales_df = df[
    (~df["IsCancelled"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

print("Raw rows:", f"{len(df):,}")
print("Sales-analysis rows:", f"{len(sales_df):,}")

Raw rows: 536,641
Sales-analysis rows: 524,878


In [17]:
# Create customer-analysis dataset
customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

print("Sales transactions:", f"{len(sales_df):,}")
print("Customer transactions:", f"{len(customer_df):,}")
print("Unique customers:", customer_df["CustomerID"].nunique())

Sales transactions: 524,878
Customer transactions: 392,692
Unique customers: 4338


In [24]:
# Save the cleaned datasets
sales_file = PROCESSED_DIR / "retail_sales_cleaned.csv"
customer_file = PROCESSED_DIR / "retail_customer_cleaned.csv"

sales_df.to_csv(sales_file, index=False)
customer_df.to_csv(customer_file, index=False)

print("Files saved successfully.")
print(sales_file)
print(customer_file)

Files saved successfully.
C:\Users\tanya\Retail_Sales_Customer_Analytics\data\processed\retail_sales_cleaned.csv
C:\Users\tanya\Retail_Sales_Customer_Analytics\data\processed\retail_customer_cleaned.csv


## Data Dictionary

| Column | Description | Data Type | Used For |
|---|---|---|---|
| InvoiceNo | Unique invoice/transaction identifier | String | Transaction analysis |
| StockCode | Product identifier | String | Product analysis |
| Description | Product description | String | Product analysis |
| Quantity | Number of units purchased | Integer | Sales analysis |
| InvoiceDate | Date and time of transaction | Datetime | Time analysis |
| UnitPrice | Price per unit | Float | Revenue calculation |
| CustomerID | Unique customer identifier | Float | Customer/RFM analysis |
| Country | Customer country | String | Geographic analysis |
| Sales | Quantity × UnitPrice | Float | Revenue analysis |
| IsCancelled | Indicates whether invoice is cancelled | Boolean | Transaction filtering |